## Task 1: Formulating the Problem

In [3]:
# ─── Imports ────────────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.metrics import (
    roc_auc_score, accuracy_score, confusion_matrix, classification_report,
    roc_curve, ConfusionMatrixDisplay
)
import joblib

plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')

RANDOM_STATE = 42
print('Libraries loaded successfully.')

In [4]:
pd.set_option("display.max_colwidth", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:.3f}".format)

data_path = "heloc_dataset_v1.csv"
df = pd.read_csv(data_path)

target_col = "RiskPerformance"   
print("df shape:", df.shape)
df.head()

# Define target (business mapping)
# original label: riskperformance in {"good", "bad"}
# we map: good = 1 (send to manual review), bad = 0 (auto reject)
target_col = "RiskPerformance"
y = (df[target_col].str.lower() == "good").astype(int)
X = df.drop(columns=[target_col])

print("df shape:", df.shape)
print("X shape:", X.shape)
print("y mean (good rate):", y.mean().round(4))
print("\nclass counts (good=1, bad=0):")
print(y.value_counts().sort_index())

In [6]:
# Simple cost model
# business interpretation:
# - false negative (fn): predicted reject (0) but actually good (1) -> we lose a good customer (opportunity cost)
# - false positive (fp): predicted manual review (1) but actually bad (0) -> extra analyst workload (labor cost)

def confusion_counts(y_true, y_pred):
    """
    return tn, fp, fn, tp in this order
    """
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    tn = np.sum((y_true == 0) & (y_pred == 0))
    fp = np.sum((y_true == 0) & (y_pred == 1))
    fn = np.sum((y_true == 1) & (y_pred == 0))
    tp = np.sum((y_true == 1) & (y_pred == 1))
    return tn, fp, fn, tp

def cost_from_counts(tn, fp, fn, tp, c_fp=1, c_fn=10):
    """
    total cost = c_fp * fp + c_fn * fn
    """
    return c_fp * fp + c_fn * fn

In [7]:
# Baseline policy: "review everyone" vs "reject everyone"
# this helps justify why we need ML at all.
y_pred_review_all = np.ones_like(y)  
y_pred_reject_all = np.zeros_like(y)  

tn, fp, fn, tp = confusion_counts(y, y_pred_review_all)
cost_review_all = cost_from_counts(tn, fp, fn, tp, c_fp=1, c_fn=10)

tn, fp, fn, tp = confusion_counts(y, y_pred_reject_all)
cost_reject_all = cost_from_counts(tn, fp, fn, tp, c_fp=1, c_fn=10)

print("\n--- baseline policies (example costs: c_fp=1, c_fn=10) ---")
print("review all -> fp, fn:", confusion_counts(y, y_pred_review_all)[1:3], "total_cost:", cost_review_all)
print("reject all -> fp, fn:", confusion_counts(y, y_pred_reject_all)[1:3], "total_cost:", cost_reject_all)

In [8]:
# auc: threshold-free ranking quality
# recall (for good=1): controls missed good customers (fn)
# precision (for good=1): controls analyst workload (fp)
task1_metrics_note = {
    "auc": "ranking quality (threshold-independent)",
    "recall_good": "how many truly good applicants we send to manual review (reduces fn)",
    "precision_good": "how many reviewed are truly good (controls fp / workload)",
}
pd.Series(task1_metrics_note, name="why_metrics")

## Task 2: Exploratory Data Analysis

In [9]:
counts = df[target_col].value_counts()
props = df[target_col].value_counts(normalize=True)

desc = df.drop(columns=[target_col]).describe().T
display(desc[["count","mean","std","min","25%","50%","75%","max"]].head(10))

counts.plot(kind="bar", rot=0, title="target distribution (riskperformance)")
plt.show()

df.info()

display(df.describe().T)

In [7]:
feature_cols = [c for c in df.columns if c != target_col]

neg_count = (df[feature_cols] < 0).sum().sort_values(ascending=False)
neg_rate  = (df[feature_cols] < 0).mean().sort_values(ascending=False)

sentinels = [-9, -8, -7]

summary = {}
for s in sentinels:
    summary[str(s)] = (df[feature_cols] == s).sum()

sentinel_df = pd.DataFrame(summary)
sentinel_df["total_sentinel"] = sentinel_df.sum(axis=1)
sentinel_df = sentinel_df.sort_values("total_sentinel", ascending=False)

X_clean = X.replace({-9: np.nan, -8: np.nan, -7: np.nan})

missing_rate = X_clean.isna().mean().sort_values(ascending=False)
display(pd.DataFrame({"missing_rate_after_sentinel_to_nan": missing_rate}).head(15))

In [8]:
fig, ax = plt.subplots(figsize=(14, 10))
corr = X_clean.corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, cmap='RdBu_r', center=0,
            vmin=-1, vmax=1, ax=ax, square=True,
            linewidths=0.5, cbar_kws={'shrink': 0.8})
ax.set_title('Feature Correlation Matrix', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('correlation_heatmap.png', dpi=120)
plt.show()

# Flag highly correlated pairs
high_corr = [(corr.columns[i], corr.columns[j], corr.iloc[i,j])
             for i in range(len(corr)) for j in range(i)
             if abs(corr.iloc[i,j]) > 0.7]
print("Highly correlated pairs (|r| > 0.7):")
for a, b, r in sorted(high_corr, key=lambda x: abs(x[2]), reverse=True):
    print(f"  {a} × {b}: {r:.3f}")

In [9]:
key_feats = [
    "ExternalRiskEstimate",
    "MSinceOldestTradeOpen",
    "MSinceMostRecentTradeOpen",
    "AverageMInFile",
    "PercentTradesNeverDelq",
    "NumTotalTrades",
]

key_feats = [c for c in key_feats if c in df.columns]
print("using:", key_feats)

for col in key_feats:
    plt.figure()
    df[df[target_col] == "Good"][col].plot(kind="hist", bins=30, alpha=0.5, label="Good")
    df[df[target_col] == "Bad"][col].plot(kind="hist", bins=30, alpha=0.5, label="Bad")
    plt.title(f"{col} distribution by riskperformance")
    plt.legend()
    plt.show()
    
print("leakage check notes:")
print("- features are bureau attributes available at application time (assumed)")
print("- avoid using any post-decision variables (none observed in provided columns)")
print("- preprocessing will be fit only on train split to avoid leakage")


In [10]:
# Explain special values: -9 = no credit bureau file, -8 = feature not applicable,
# -7 = condition never occurred (e.g. never delinquent).
# Count occurrences and show class balance.

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

data_path = "heloc_dataset_v1.csv"
df = pd.read_csv(data_path)
feat_cols = [c for c in df.columns if c != "RiskPerformance"]
y = (df["RiskPerformance"].str.lower() == "good").astype(int)

# Show special value counts per feature
print("Special value counts per feature:")
for val, label in [(-9, "no credit file"), (-8, "not applicable"), (-7, "never occurred")]:
    cols_with_val = {c: (df[c] == val).sum() for c in feat_cols if (df[c] == val).sum() > 0}
    if cols_with_val:
        print(f"  value={val} ({label}):")
        for c, cnt in cols_with_val.items():
            print(f"    {c}: {cnt}")

# Class balance bar chart (matches original plot style)
counts = df["RiskPerformance"].value_counts()
plt.figure()
plt.bar(counts.index, counts.values)
plt.title("RiskPerformance class balance")
plt.ylabel("count")
plt.show()
print("Good rate:", y.mean().round(4))
print("=> Classes nearly balanced (52% Bad / 48% Good). No resampling needed.")


In [11]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
counts = df['RiskPerformance'].value_counts()
axes[0].pie(counts, labels=counts.index, autopct='%1.1f%%',
            colors=['#2ecc71','#e74c3c'], startangle=90)
axes[0].set_title('Class Distribution')

axes[1].bar(counts.index, counts.values, color=['#2ecc71','#e74c3c'])
axes[1].set_title('Class Counts')
for i, v in enumerate(counts.values):
    axes[1].text(i, v + 50, str(v), ha='center', fontweight='bold')
plt.tight_layout()
plt.savefig('class_balance.png', dpi=120)
plt.show()
print(f"Class balance ratio: {counts.min()/counts.max():.3f} (well balanced, no resampling needed)")

In [12]:
# Missing values statistics
missing = df.isnull().sum()
missing_pct = (missing / len(df)) * 100

missing_df = pd.DataFrame({
    'Missing_Count': missing,
    'Missing_Percentage': missing_pct
}).sort_values('Missing_Percentage', ascending=False)

print("Missing Values Summary:")
print("=" * 60)
print(missing_df[missing_df['Missing_Count'] > 0])

total_missing = missing_df['Missing_Count'].sum()
print(f"\nTotal Missing Values: {total_missing}")
print(f"Overall Missing Percentage: {(total_missing / df.size) * 100:.2f}%")

In [13]:
# Detect special values
# Explain special values: -9 = no credit bureau file, -8 = feature not applicable,
# -7 = condition never occurred (e.g. never delinquent).
special_values = [-9, -8, -7]
special_value_stats = []

for col in df.columns:
    if df[col].dtype in ['int64', 'float64']:
        special_count = df[col].isin(special_values).sum()
        if special_count > 0:
            special_pct = (special_count / len(df)) * 100
            special_value_stats.append({
                'Feature': col,
                'Special_Value_Count': special_count,
                'Special_Value_Percentage': special_pct
            })

if special_value_stats:
    special_df = pd.DataFrame(special_value_stats).sort_values('Special_Value_Percentage', ascending=False)
    print("Special Values (-9, -8, -7) Statistics:")
    print("=" * 60)
    print(special_df)
    
    print(f"\n⚠️  {len(special_df)} features contain special values")
    print("Recommendation: Replace special values with NaN, then handle as missing values")
else:
    print("✓ No special values detected")

# Data Preprocessing 

## Task 3: ML Model

In [14]:
# ─── Preprocessing ────────────────────────────────────────────────────────────
df_model = df.copy()
df_model['target'] = (df_model['RiskPerformance'] == 'Bad').astype(int)

X = df_model[feature_cols].copy()
y = df_model['target'].copy()

# Replace sentinel values with NaN FIRST (before split)
for col in X.columns:
    X[col] = X[col].replace([-7, -8, -9], np.nan)

# Split BEFORE computing medians (prevents data leakage)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

# Compute medians on TRAINING SET ONLY
medians = X_train.median()

# Fill NaN using training medians (apply same values to test set)
X_train = X_train.fillna(medians)
X_test  = X_test.fillna(medians)   # test set uses train medians, not its own

print(f'Training set: {X_train.shape[0]} samples')
print(f'Test set:     {X_test.shape[0]} samples')
print(f'Bad rate (train): {y_train.mean():.3f} | Bad rate (test): {y_test.mean():.3f}')
print(f'\nMedians computed on training set only (no leakage).')
print(f'Example — ExternalRiskEstimate median (train): {medians["ExternalRiskEstimate"]:.1f}')

In [15]:
# ─── Step 1: Candidate Model Comparison (5-Fold CV) ─────────────────────────
# We compare four candidate models using cross-validated AUC-ROC.
# Models immediately ruled out:
#   - Logistic Regression: assumes linear decision boundary, unsuitable for
#     complex nonlinear credit interactions
#   - Decision Tree: high variance, prone to overfitting on tabular data
# Ensemble methods (Random Forest, GBM) are expected to outperform.

candidate_models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=RANDOM_STATE),
    'Decision Tree':       DecisionTreeClassifier(max_depth=10, random_state=RANDOM_STATE),
    'Random Forest':       RandomForestClassifier(n_estimators=100, random_state=RANDOM_STATE, n_jobs=-1),
    'Gradient Boosting':   GradientBoostingClassifier(n_estimators=100, random_state=RANDOM_STATE),
}

cv_results = {}
for name, model in candidate_models.items():
    scores = cross_val_score(model, X_train, y_train, cv=5, scoring='roc_auc')
    cv_results[name] = scores
    print(f'{name:25s}  AUC: {scores.mean():.4f} ± {scores.std():.4f}')

fig, ax = plt.subplots(figsize=(10, 5))
means  = [v.mean() for v in cv_results.values()]
stds   = [v.std()  for v in cv_results.values()]
colors = ['#95a5a6', '#95a5a6', '#95a5a6', '#e74c3c']
bars   = ax.bar(cv_results.keys(), means, yerr=stds, capsize=5,
                color=colors, edgecolor='white', linewidth=1.5)
ax.set_ylim(0.60, 0.85)
ax.set_ylabel('5-Fold CV AUC-ROC')
ax.set_title('Model Comparison — Cross-Validation AUC', fontsize=14, fontweight='bold')
for bar, mean in zip(bars, means):
    ax.text(bar.get_x() + bar.get_width()/2, mean + 0.005, f'{mean:.4f}',
            ha='center', fontsize=11, fontweight='bold')
ax.axhline(0.75, color='navy', linestyle='--', alpha=0.5, label='Minimum threshold (0.75)')
ax.legend()
plt.tight_layout()
plt.savefig('model_comparison.png', dpi=120, bbox_inches='tight')
plt.show()


In [16]:
# ─── Step 2: Hyperparameter Tuning — Gradient Boosting (Grid Search) ────────
# GBM had the highest CV AUC. We now tune its hyperparameters systematically.
from sklearn.model_selection import GridSearchCV

param_grid = {
    'n_estimators':      [100, 200, 300],
    'max_depth':         [3, 4, 5],
    'learning_rate':     [0.03, 0.05, 0.1],
    'min_samples_split': [30, 50, 80],
}

grid_search = GridSearchCV(
    GradientBoostingClassifier(subsample=0.8, random_state=RANDOM_STATE),
    param_grid,
    cv=5,
    scoring='roc_auc',
    n_jobs=-1,
    verbose=1
)
grid_search.fit(X_train, y_train)

gbm_best  = grid_search.best_estimator_
gbm_prob  = gbm_best.predict_proba(X_test)[:, 1]
gbm_auc   = roc_auc_score(y_test, gbm_prob)

print(f'Best CV AUC  : {grid_search.best_score_:.4f}')
print(f'Best params  : {grid_search.best_params_}')
print(f'Test AUC     : {gbm_auc:.4f}')

In [17]:
# ─── Step 3: XGBoost vs Tuned GBM — Select Final Model ─────────────────────
# XGBoost is an optimized GBM implementation with built-in regularization.
# We compare it against the tuned GBM and select the better performer.
from xgboost import XGBClassifier

xgb_model = XGBClassifier(
    n_estimators     = 300,
    max_depth        = 4,
    learning_rate    = 0.05,
    subsample        = 0.8,
    colsample_bytree = 0.8,
    min_child_weight = 30,
    eval_metric      = 'auc',
    random_state     = RANDOM_STATE,
    verbosity        = 0
)

xgb_cv   = cross_val_score(xgb_model, X_train, y_train, cv=5, scoring='roc_auc')
xgb_model.fit(X_train, y_train)
xgb_prob = xgb_model.predict_proba(X_test)[:, 1]
xgb_auc  = roc_auc_score(y_test, xgb_prob)

print(f'XGBoost  CV AUC   : {xgb_cv.mean():.4f} ± {xgb_cv.std():.4f}')
print(f'GBM      CV AUC   : {grid_search.best_score_:.4f}')
print(f'XGBoost  Test AUC : {xgb_auc:.4f}')
print(f'GBM      Test AUC : {gbm_auc:.4f}')

# Select the better model
if xgb_auc >= gbm_auc:
    final_model = xgb_model
    y_prob      = xgb_prob
    model_name  = 'XGBoost'
else:
    final_model = gbm_best
    y_prob      = gbm_prob
    model_name  = 'Gradient Boosting'

y_pred = final_model.predict(X_test)
print(f'\n✅ Final model selected: {model_name}')
print(f'   Test AUC-ROC  : {roc_auc_score(y_test, y_prob):.4f}')
print(f'   Test Accuracy : {accuracy_score(y_test, y_pred):.4f}')
print(f'\nClassification Report:')
print(classification_report(y_test, y_pred, target_names=['Good (0)', 'Bad (1)']))


In [18]:
# ─── Step 4: Model Evaluation — ROC Curve & Confusion Matrix ────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ── ROC Curve ────────────────────────────────────────────────────────────────
fpr, tpr, _ = roc_curve(y_test, y_prob)
auc_score   = roc_auc_score(y_test, y_prob)

axes[0].plot(fpr, tpr, color='#e74c3c', lw=2,
             label=f'{model_name} (AUC = {auc_score:.4f})')
axes[0].plot([0, 1], [0, 1], color='gray', linestyle='--', lw=1,
             label='Random classifier')
axes[0].fill_between(fpr, tpr, alpha=0.1, color='#e74c3c')
axes[0].set_xlabel('False Positive Rate')
axes[0].set_ylabel('True Positive Rate')
axes[0].set_title(f'ROC Curve — {model_name}', fontsize=13, fontweight='bold')
axes[0].legend(fontsize=11)

# ── Confusion Matrix ──────────────────────────────────────────────────────────
cm = confusion_matrix(y_test, y_pred)
ConfusionMatrixDisplay(confusion_matrix=cm,
                       display_labels=['Good', 'Bad']).plot(
    ax=axes[1], colorbar=False, cmap='Blues')
axes[1].set_title('Confusion Matrix — Test Set', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.savefig('model_evaluation.png', dpi=120, bbox_inches='tight')
plt.show()

tn, fp, fn, tp = cm.ravel()
print(f'TN — Good correctly sent to officer : {tn}')
print(f'FP — Good wrongly auto-rejected     : {fp}   ← business FN (lost customer)')
print(f'FN — Bad  wrongly sent to officer   : {fn}   ← business FP (wasted review)')
print(f'TP — Bad  correctly auto-rejected   : {tp}')


In [19]:
# ─── Step 5: Feature Importances ────────────────────────────────────────────
fi_df = pd.DataFrame({
    'Feature'   : feature_cols,
    'Importance': final_model.feature_importances_
}).sort_values('Importance', ascending=True)

fig, ax = plt.subplots(figsize=(10, 8))
colors = ['#e74c3c' if imp > 0.04 else '#3498db' for imp in fi_df['Importance']]
ax.barh(fi_df['Feature'], fi_df['Importance'], color=colors, edgecolor='white')
ax.set_xlabel('Feature Importance (Gain)')
ax.set_title(f'{model_name} — Feature Importances', fontsize=14, fontweight='bold')
ax.axvline(0.04, color='black', linestyle='--', alpha=0.5, label='Top-feature threshold')
ax.legend()
plt.tight_layout()
plt.savefig('feature_importance.png', dpi=120, bbox_inches='tight')
plt.show()

print('Top 5 most important features:')
print(fi_df.tail(5)[['Feature', 'Importance']].to_string(index=False))


In [20]:
# ─── Step 6: Optimal Decision Threshold (Business Cost Minimization) ────────
#
# Label mapping: Bad = 1, Good = 0  →  y_prob = P(Bad)
#   y_prob >= threshold  →  predicted Bad  →  auto-rejected (DENIED)
#   y_prob <  threshold  →  predicted Good →  sent to officer (APPROVED)
#
# sklearn confusion_matrix (negative=Good=0, positive=Bad=1):
#   tn: true Good, pred Good  → correctly approved          ✓
#   fp: true Good, pred Bad   → Good wrongly rejected       → business FN  (cost = C_FN)
#   fn: true Bad,  pred Good  → Bad wrongly sent to officer → business FP  (cost = C_FP)
#   tp: true Bad,  pred Bad   → correctly rejected          ✓
#
# Cost assumptions (c_FN = 3 × c_FP reflects that losing a good customer
# is ~3× more costly than one unnecessary officer review):
C_FP, C_FN = 1, 3

fpr_t, tpr_t, thresholds = roc_curve(y_test, y_prob)

costs = []
for t in thresholds:
    y_pred_t = (y_prob >= t).astype(int)
    tn_t, fp_t, fn_t, tp_t = confusion_matrix(y_test, y_pred_t).ravel()
    costs.append(fn_t * C_FP + fp_t * C_FN)   # business FP + business FN

best_idx       = np.argmin(costs)
best_threshold = float(thresholds[best_idx])
best_cost      = costs[best_idx]
default_idx    = np.argmin(np.abs(thresholds - 0.5))
default_cost   = costs[default_idx]

print('=' * 55)
print('        THRESHOLD OPTIMIZATION RESULTS')
print('=' * 55)
print(f'  Cost assumptions : c_FP = {C_FP}  |  c_FN = {C_FN}')
print('-' * 55)
print(f'  Default threshold (0.5)  cost : {default_cost:>6,}')
print(f'  Optimal threshold ({best_threshold:.3f}) cost : {best_cost:>6,}')
print(f'  Cost reduction           : {default_cost - best_cost:>+6,} '
      f'({(default_cost - best_cost) / default_cost * 100:+.1f}%)')
print('=' * 55)

y_pred_optimal  = (y_prob >= best_threshold).astype(int)
tn_o, fp_o, fn_o, tp_o = confusion_matrix(y_test, y_pred_optimal).ravel()
print(f'\nConfusion matrix at optimal threshold ({best_threshold:.3f}):')
print(f'  Good correctly sent to officer : {tn_o}')
print(f'  Good wrongly auto-rejected     : {fp_o}  ← business FN')
print(f'  Bad  wrongly sent to officer   : {fn_o}  ← business FP')
print(f'  Bad  correctly auto-rejected   : {tp_o}')

# ── Plot ─────────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(thresholds, costs, color='#e74c3c', lw=2, label='Business cost')
ax.axvline(0.5, color='gray', linestyle='--', alpha=0.8,
           label=f'Default (0.5) — cost = {default_cost:,}')
ax.axvline(best_threshold, color='navy', linestyle='--', alpha=0.9,
           label=f'Optimal ({best_threshold:.3f}) — cost = {best_cost:,}')
ax.scatter([best_threshold], [best_cost], color='navy', zorder=5, s=80)
ax.annotate(f'Optimal\n({best_threshold:.3f}, {best_cost:,})',
            xy=(best_threshold, best_cost),
            xytext=(best_threshold + 0.06, best_cost + 200),
            fontsize=9, color='navy',
            arrowprops=dict(arrowstyle='->', color='navy'))
ax.set_xlabel('Decision Threshold  [P(Bad) cutoff]', fontsize=11)
ax.set_ylabel('Total Business Cost', fontsize=11)
ax.set_title('Cost vs. Threshold — Finding the Optimal Decision Cutoff',
             fontsize=13, fontweight='bold')
ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig('threshold_optimization.png', dpi=120, bbox_inches='tight')
plt.show()


In [21]:
# ─── Cost Savings Summary ─────────────────────────────────────────────────────
n_test = len(y_test)
n_good = (y_test == 0).sum()
n_bad  = (y_test == 1).sum()

# Baseline costs on test set
cost_review_all_test = n_bad  * C_FP   # send everyone → bad ones waste time
cost_reject_all_test = n_good * C_FN   # reject everyone → good ones lost

print("=" * 50)
print("       FINAL COST COMPARISON (Test Set)")
print("=" * 50)
print(f"  Reject All  : {cost_reject_all_test:>6,} units")
print(f"  Review All  : {cost_review_all_test:>6,} units")
print(f"  ML (t=0.5)  : {default_cost:>6,} units")
print(f"  ML (t={best_threshold:.3f}): {best_cost:>6,} units  ← best")
print("-" * 50)
print(f"  Saving vs Review All : {cost_review_all_test - best_cost:+,} ({(cost_review_all_test-best_cost)/cost_review_all_test*100:+.1f}%)")
print("=" * 50)

In [22]:
from sklearn.model_selection import learning_curve

train_sizes, train_scores, val_scores = learning_curve(
    final_model, X_train, y_train,
    cv=3, scoring='roc_auc',
    train_sizes=np.linspace(0.2, 1.0, 5),
    n_jobs=-1
)
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(train_sizes, train_scores.mean(1), label='Train AUC', color='#e74c3c')
ax.plot(train_sizes, val_scores.mean(1), label='Val AUC', color='navy')
ax.fill_between(train_sizes, train_scores.mean(1)-train_scores.std(1),
                train_scores.mean(1)+train_scores.std(1), alpha=0.1, color='#e74c3c')
ax.fill_between(train_sizes, val_scores.mean(1)-val_scores.std(1),
                val_scores.mean(1)+val_scores.std(1), alpha=0.1, color='navy')
ax.set_xlabel('Training Set Size')
ax.set_ylabel('AUC-ROC')
ax.set_title('Learning Curve — No Overfitting', fontweight='bold')
ax.legend()
plt.tight_layout()
plt.savefig('learning_curve.png', dpi=120)
plt.show()

In [23]:
# ─── Save All Artifacts for Streamlit App ────────────────────────────────────
import joblib

joblib.dump(final_model,            'heloc_model.joblib')
joblib.dump(medians,                'heloc_medians.joblib')
joblib.dump(list(X_train.columns),  'feature_cols.joblib')
joblib.dump(best_threshold,         'heloc_threshold.joblib')

print('✅ heloc_model.joblib      — trained model')
print('✅ heloc_medians.joblib    — training-set medians for imputation')
print('✅ feature_cols.joblib     — feature name list (order matters)')
print(f'✅ heloc_threshold.joblib  — optimal threshold ({best_threshold:.3f})')


## Task 4: Explanations

In [24]:
# ─── Task 4: Explanations  ─────────────────
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from xgboost import plot_importance

In [25]:
# ── Global: Feature Importance (3 methods) ────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

for ax, imp_type in zip(axes, ['weight', 'gain', 'cover']):
    plot_importance(final_model, importance_type=imp_type,
                    ax=ax, max_num_features=10,
                    title=f'Feature Importance ({imp_type})',
                    show_values=False)
plt.tight_layout()
plt.savefig('feature_importance_xgb.png', dpi=120, bbox_inches='tight')
plt.show()

In [26]:
# ── Individual: Denial explanation (rule-based using feature importance) ───────
IMPROVEMENT_TIPS = {
    'ExternalRiskEstimate':
        'Pay bills on time and reduce debt to improve your credit score.',
    'PercentTradesNeverDelq':
        'Keep all accounts current — avoid any late payments.',
    'NetFractionRevolvingBurden':
        'Pay down credit card balances to below 30% of your limits.',
    'NumBank2NatlTradesWHighUtilization':
        'Reduce balances on high-utilization accounts.',
    'NumTrades60Ever2DerogPubRec':
        'Avoid future delinquencies; past late payments hurt approval odds.',
    'NumTrades90Ever2DerogPubRec':
        'Build a consistent on-time payment history going forward.',
    'MaxDelqEver':
        'Continue building a clean payment record over time.',
    'MaxDelq2PublicRecLast12M':
        'Bring all accounts current immediately.',
    'MSinceMostRecentDelq':
        'Maintain clean payment history; time heals past delinquencies.',
    'NumInqLast6M':
        'Avoid applying for multiple credit products at once.',
    'NetFractionInstallBurden':
        'Pay down installment loan balances.',
    'NumRevolvingTradesWBalance':
        'Pay off some revolving accounts entirely.',
    'MSinceMostRecentInqexcl7days':
        'Avoid applying for new credit products — recent inquiries signal financial stress.',
}

def generate_denial_letter(applicant_row, model, feature_cols, top_n=3):
    """
    Generate a denial letter for one applicant.
    applicant_row: pd.Series with feature values (after preprocessing)
    """
    # Get feature importances (gain = most meaningful)
    scores = model.get_booster().get_score(importance_type='gain')
    
    # Rank features by importance, filter to those with bad values
    importance_df = pd.DataFrame([
        {'feature': f, 'importance': scores.get(f, 0)}
        for f in feature_cols
    ]).sort_values('importance', ascending=False)
    
    top_features = importance_df.head(top_n)['feature'].tolist()
    
    print("=" * 60)
    print("    HELOC APPLICATION — NOTICE OF ADVERSE ACTION")
    print("=" * 60)
    print("\nDecision: APPLICATION DENIED\n")
    print("Your application did not meet our minimum criteria for")
    print("initial screening. Principal reasons for this decision:\n")
    
    for i, feat in enumerate(top_features, 1):
        val = applicant_row[feat]
        tip = IMPROVEMENT_TIPS.get(feat, 'Work on improving this factor.')
        label = feat.replace('_', ' ')
        print(f"  {i}. {label}")
        print(f"     Current value : {val:.0f}")
        print(f"     How to improve: {tip}\n")
    
    print("=" * 60)
    print("You may reapply after addressing the factors listed above.")
    print("Simon Bank of Rochester® | Equal Housing Lender")
    print("=" * 60)

In [27]:
# ── Find a denied applicant in the test set ───────────────────────────────────
y_pred_at_threshold = (y_prob >= best_threshold).astype(int)
denied_indices = np.where(y_pred_at_threshold == 1)[0]  # Bad = 1 = denied

sample_idx = denied_indices[0]
applicant  = X_test.iloc[sample_idx]

print(f"Example: Applicant #{sample_idx} (Predicted: DENIED, "
      f"P(Bad)={y_prob[sample_idx]:.3f})\n")
generate_denial_letter(applicant, final_model, feature_cols, top_n=3)

In [ ]:
# ── Probability distribution plot ─────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(y_prob[y_test == 0], bins=40, alpha=0.6, color='#2ecc71',
        label='Good applicants', density=True)
ax.hist(y_prob[y_test == 1], bins=40, alpha=0.6, color='#e74c3c',
        label='Bad applicants', density=True)
ax.axvline(best_threshold, color='navy', linestyle='--', lw=2,
           label=f'Decision threshold ({best_threshold:.3f})')
ax.set_xlabel('P(Bad) — Model Score', fontsize=11)
ax.set_ylabel('Density', fontsize=11)
ax.set_title('Score Distribution: Good vs Bad Applicants', fontsize=13,
             fontweight='bold')
ax.legend(fontsize=10)
plt.tight_layout()
plt.savefig('score_distribution.png', dpi=120, bbox_inches='tight')
plt.show()